# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_06 — Monte Carlo GBM and Fat-Tailed Return Simulation

### Research question

How can Monte Carlo simulation price derivatives under geometric Brownian motion, and how do fat-tailed return models change tail risk, option prices, and model interpretation?

This notebook builds a Monte Carlo pricing and risk simulation framework.

It covers:

1. exact terminal simulation under risk-neutral GBM;
2. path simulation for future path-dependent pricing;
3. Monte Carlo option pricing with confidence intervals;
4. convergence rate and standard error;
5. antithetic variates;
6. control variates;
7. risk-neutral martingale checks;
8. fat-tailed return simulations;
9. normal-mixture shocks;
10. truncated Student-t stress shocks;
11. tail-risk diagnostics: VaR and expected shortfall;
12. option price sensitivity under fat tails;
13. why not every heavy-tailed distribution is automatically valid for arbitrage-free pricing;
14. saved Monte Carlo outputs and diagnostics.

The key idea is:

> Monte Carlo is not only a pricing method. It is a laboratory for understanding model assumptions, numerical error, and tail risk.

## 1. Why Monte Carlo?

Closed-form formulas are rare.

The Black-Scholes-Merton formula prices simple European options under very specific assumptions. But many realistic problems involve:

- path-dependent payoffs;
- stochastic volatility;
- jumps;
- fat tails;
- early-exercise approximations;
- transaction costs;
- portfolio-level nonlinear risk;
- high-dimensional risk factors.

Monte Carlo simulation prices a derivative by estimating the discounted expected payoff under a pricing measure:

$$
V_0 = e^{-rT}\mathbb E^{\mathbb Q}[g(S_T)]
$$
or, for path-dependent derivatives:

$$
V_0 = e^{-rT}\mathbb E^{\mathbb Q}[g(S_{t_0},S_{t_1},\dots,S_T)]
$$
The method is flexible but introduces statistical error:

$$
\begin{aligned}
\text{standard error} &= \frac{\hat{\sigma}_{\text{payoff}}}{\sqrt{M}}
\end{aligned}
$$
where $M$ is the number of simulated paths.

So Monte Carlo convergence is slow:

$$
\text{error} = O(M^{-1/2})
$$

## 2. Risk-neutral GBM

Under the risk-neutral measure $\mathbb Q$, the Black-Scholes-Merton stock process is:

$$
dS_t = (r-q)S_tdt + \sigma S_tdW_t^{\mathbb Q}
$$
where:

- $r$ is the risk-free rate;
- $q$ is the continuous dividend yield;
- $\sigma$ is volatility;
- $W_t^{\mathbb Q}$ is Brownian motion under the pricing measure.

The exact terminal solution is:

$$
\begin{aligned}
S_T &= S_0 \exp \Bigg[ (r-q-\frac{1}{2}\sigma^2)T \\
&\quad + \sigma\sqrt{T}Z \Bigg]
\end{aligned}
$$
where:

$$
Z \sim \mathcal N(0,1)
$$
This exact solution avoids time-discretisation error for terminal European payoffs.

## 3. Fat-tailed simulation warning

Financial returns often have heavier tails than Gaussian returns.

However, there is an important modelling distinction:

### Risk simulation

For stress testing, VaR, expected shortfall, or empirical scenario analysis, one may simulate heavy-tailed returns directly.

### Risk-neutral pricing

For arbitrage-free pricing, the discounted stock price should satisfy:

$$
\begin{aligned}
\mathbb E^{\mathbb Q} \left[ e^{-rT}S_T \right] &= S_0
\end{aligned}
$$
If we write:

$$
S_T = S_0 \exp((r-q)T + X - m_X)
$$
then the martingale correction must be:

$$
m_X = \log \mathbb E[e^X]
$$
Some heavy-tailed distributions, such as the unbounded Student-t distribution, do not have a finite moment generating function. That means $\mathbb E[e^X]$ can be infinite.

So a raw Student-t log-return shock is not automatically a valid risk-neutral stock model.

In this notebook:

1. **GBM** is a clean arbitrage-free pricing baseline.
2. **Normal mixture shocks** have finite exponential moments and can be martingale-corrected analytically.
3. **Truncated Student-t shocks** are used as a stress-testing model with empirical martingale correction.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class MonteCarloConfig:
    """
    Monte Carlo model configuration.
    """
    s0: float = 100.0
    strike: float = 100.0
    maturity: float = 1.0
    risk_free_rate: float = 0.04
    dividend_yield: float = 0.00
    volatility: float = 0.20
    n_paths: int = 200_000
    n_steps: int = 252
    seed: int = 42


config = MonteCarloConfig()
config

## 5. Black-Scholes-Merton reference functions

We use the analytical BSM formula as a reference for European option prices under GBM.

This lets us measure Monte Carlo bias and sampling error.

In [ ]:
def normal_cdf(x: float | np.ndarray) -> float | np.ndarray:
    """
    Standard normal CDF.
    """
    x = np.asarray(x)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def bsm_price(
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    volatility: float,
    option_type: str
) -> float:
    """
    Black-Scholes-Merton European option price.
    """
    if maturity <= 0:
        if option_type == "call":
            return max(spot - strike, 0.0)
        if option_type == "put":
            return max(strike - spot, 0.0)
        raise ValueError("option_type must be 'call' or 'put'.")

    d1 = (
        log(spot / strike)
        + (risk_free_rate - dividend_yield + 0.5 * volatility ** 2) * maturity
    ) / (volatility * sqrt(maturity))

    d2 = d1 - volatility * sqrt(maturity)

    discounted_spot = spot * exp(-dividend_yield * maturity)
    discounted_strike = strike * exp(-risk_free_rate * maturity)

    if option_type == "call":
        return float(discounted_spot * normal_cdf(d1) - discounted_strike * normal_cdf(d2))

    if option_type == "put":
        return float(discounted_strike * normal_cdf(-d2) - discounted_spot * normal_cdf(-d1))

    raise ValueError("option_type must be 'call' or 'put'.")


bsm_call_reference = bsm_price(
    spot=config.s0,
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    volatility=config.volatility,
    option_type="call"
)

bsm_put_reference = bsm_price(
    spot=config.s0,
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield,
    volatility=config.volatility,
    option_type="put"
)

pd.Series({
    "bsm_call_reference": bsm_call_reference,
    "bsm_put_reference": bsm_put_reference
})

## 6. Exact terminal GBM simulation

For a European payoff depending only on $S_T$, exact terminal simulation is enough.

The terminal stock price is:

$$
\begin{aligned}
S_T &= S_0 \exp \Bigg[ (r-q-\frac{1}{2}\sigma^2)T \\
&\quad + \sigma\sqrt{T}Z \Bigg]
\end{aligned}
$$

In [ ]:
def simulate_gbm_terminal_prices(
    config: MonteCarloConfig,
    n_paths: int | None = None,
    seed: int | None = None
) -> pd.DataFrame:
    """
    Simulate exact terminal prices under risk-neutral GBM.
    """
    if n_paths is None:
        n_paths = config.n_paths

    if seed is None:
        seed = config.seed

    local_rng = np.random.default_rng(seed)

    z = local_rng.standard_normal(n_paths)

    log_return = (
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility ** 2)
        * config.maturity
        + config.volatility * sqrt(config.maturity) * z
    )

    terminal_price = config.s0 * np.exp(log_return)

    return pd.DataFrame({
        "terminal_price": terminal_price,
        "log_return": log_return,
        "z": z
    })


gbm_terminal = simulate_gbm_terminal_prices(config)

gbm_terminal.head()

## 7. Monte Carlo option pricing

For a payoff $g(S_T)$, the Monte Carlo estimator is:

$$
\begin{aligned}
\hat V_0 &= e^{-rT} \frac{1}{M} \sum_{m=1}^{M}g(S_T^{(m)})
\end{aligned}
$$
A 95% confidence interval is:

$$
\hat V_0
\pm
1.96
\frac{s_{\text{discounted payoff}}}{\sqrt M}
$$

In [ ]:
def vanilla_payoff(terminal_price: np.ndarray, strike: float, option_type: str) -> np.ndarray:
    """
    Vanilla call or put payoff.
    """
    if option_type == "call":
        return np.maximum(terminal_price - strike, 0.0)

    if option_type == "put":
        return np.maximum(strike - terminal_price, 0.0)

    raise ValueError("option_type must be 'call' or 'put'.")


def monte_carlo_price_from_terminal(
    terminal_price: np.ndarray,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    option_type: str
) -> dict:
    """
    Price a vanilla option from terminal simulated prices.
    """
    payoff = vanilla_payoff(terminal_price, strike, option_type)
    discounted_payoff = np.exp(-risk_free_rate * maturity) * payoff

    price = float(np.mean(discounted_payoff))
    standard_error = float(np.std(discounted_payoff, ddof=1) / np.sqrt(len(discounted_payoff)))

    return {
        "price": price,
        "standard_error": standard_error,
        "lower_95": price - 1.96 * standard_error,
        "upper_95": price + 1.96 * standard_error,
        "n_paths": len(discounted_payoff),
        "option_type": option_type
    }


mc_call = monte_carlo_price_from_terminal(
    terminal_price=gbm_terminal["terminal_price"].to_numpy(),
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    option_type="call"
)

mc_put = monte_carlo_price_from_terminal(
    terminal_price=gbm_terminal["terminal_price"].to_numpy(),
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    option_type="put"
)

pd.DataFrame([
    {"reference": bsm_call_reference, **mc_call, "error_vs_bsm": mc_call["price"] - bsm_call_reference},
    {"reference": bsm_put_reference, **mc_put, "error_vs_bsm": mc_put["price"] - bsm_put_reference},
])

## 8. Martingale check

Under the risk-neutral measure:

$$
\mathbb E^{\mathbb Q}[S_T] = S_0e^{(r-q)T}
$$
Equivalently:

$$
\begin{aligned}
e^{-rT}\mathbb E^{\mathbb Q}[S_T] &= S_0e^{-qT}
\end{aligned}
$$
A simulation should approximately satisfy this.

In [ ]:
def martingale_check_terminal(
    terminal_price: np.ndarray,
    spot: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float
) -> pd.Series:
    """
    Check risk-neutral martingale condition for terminal prices.
    """
    expected_terminal = spot * exp((risk_free_rate - dividend_yield) * maturity)
    simulated_mean = float(np.mean(terminal_price))

    discounted_expected = exp(-risk_free_rate * maturity) * simulated_mean
    discounted_theoretical = spot * exp(-dividend_yield * maturity)

    return pd.Series({
        "simulated_mean_terminal_price": simulated_mean,
        "theoretical_mean_terminal_price": expected_terminal,
        "mean_terminal_error": simulated_mean - expected_terminal,
        "discounted_simulated_mean": discounted_expected,
        "discounted_theoretical_spot": discounted_theoretical,
        "discounted_martingale_error": discounted_expected - discounted_theoretical
    })


martingale_check_gbm = martingale_check_terminal(
    terminal_price=gbm_terminal["terminal_price"].to_numpy(),
    spot=config.s0,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield
)

martingale_check_gbm

## 9. Monte Carlo convergence

Monte Carlo error falls at rate:

$$
O(M^{-1/2})
$$
To reduce error by a factor of 10, the number of paths must increase by roughly 100.

This is why variance reduction methods matter.

In [ ]:
def convergence_experiment_gbm(
    config: MonteCarloConfig,
    path_grid: list[int],
    option_type: str,
    seed: int = 123
) -> pd.DataFrame:
    """
    Price option with increasing number of Monte Carlo paths.
    """
    rows = []

    reference = bsm_price(
        spot=config.s0,
        strike=config.strike,
        maturity=config.maturity,
        risk_free_rate=config.risk_free_rate,
        dividend_yield=config.dividend_yield,
        volatility=config.volatility,
        option_type=option_type
    )

    for n_paths in path_grid:
        sim = simulate_gbm_terminal_prices(config, n_paths=n_paths, seed=seed)
        result = monte_carlo_price_from_terminal(
            terminal_price=sim["terminal_price"].to_numpy(),
            strike=config.strike,
            maturity=config.maturity,
            risk_free_rate=config.risk_free_rate,
            option_type=option_type
        )

        rows.append({
            "n_paths": n_paths,
            "option_type": option_type,
            "mc_price": result["price"],
            "standard_error": result["standard_error"],
            "reference_bsm": reference,
            "absolute_error": abs(result["price"] - reference),
            "ci_width_95": result["upper_95"] - result["lower_95"]
        })

    return pd.DataFrame(rows)


path_grid = [1_000, 2_500, 5_000, 10_000, 25_000, 50_000, 100_000, 200_000]

convergence_call = convergence_experiment_gbm(
    config=config,
    path_grid=path_grid,
    option_type="call",
    seed=123
)

convergence_call

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(convergence_call["n_paths"], convergence_call["absolute_error"], marker="o", label="Absolute error")
plt.plot(convergence_call["n_paths"], convergence_call["standard_error"], marker="x", label="Standard error")
plt.xscale("log")
plt.yscale("log")
plt.title("Monte Carlo Convergence for GBM Call Price")
plt.xlabel("Number of paths")
plt.ylabel("Error scale")
plt.legend()
plt.show()

## 10. Path simulation

For European vanilla options, exact terminal simulation is enough.

For path-dependent options, we need the full path:

$$
S_{t_0},S_{t_1},\dots,S_T
$$
Examples:

- Asian options;
- barrier options;
- lookback options;
- drawdown derivatives;
- dynamic hedging simulations.

We simulate paths using the exact GBM transition over each time step.

In [ ]:
def simulate_gbm_paths(
    config: MonteCarloConfig,
    n_paths: int | None = None,
    n_steps: int | None = None,
    seed: int | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """
    Simulate full GBM price paths using exact discrete transitions.
    """
    if n_paths is None:
        n_paths = config.n_paths

    if n_steps is None:
        n_steps = config.n_steps

    if seed is None:
        seed = config.seed

    local_rng = np.random.default_rng(seed)

    dt = config.maturity / n_steps
    time_grid = np.linspace(0.0, config.maturity, n_steps + 1)

    z = local_rng.standard_normal(size=(n_paths, n_steps))

    log_increments = (
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility ** 2) * dt
        + config.volatility * sqrt(dt) * z
    )

    log_paths = np.zeros((n_paths, n_steps + 1))
    log_paths[:, 0] = log(config.s0)
    log_paths[:, 1:] = log(config.s0) + np.cumsum(log_increments, axis=1)

    price_paths = np.exp(log_paths)

    return time_grid, price_paths


time_grid, gbm_paths = simulate_gbm_paths(
    config=config,
    n_paths=2_000,
    n_steps=config.n_steps,
    seed=2026
)

gbm_paths.shape

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(60):
    plt.plot(time_grid, gbm_paths[i], alpha=0.35)

plt.title("GBM Sample Paths")
plt.xlabel("Time")
plt.ylabel("Price")
plt.show()

## 11. Antithetic variates

Antithetic variates reduce variance by pairing each normal shock $Z$ with $-Z$.

If one path has a high terminal price, its antithetic path tends to have a lower terminal price.

This can reduce payoff variance for many monotonic payoffs.

For terminal GBM:

$$
S_T(Z)
\quad \text{and} \quad
S_T(-Z)
$$
are simulated together.

In [ ]:
def simulate_gbm_terminal_antithetic(
    config: MonteCarloConfig,
    n_pairs: int,
    seed: int = 42
) -> pd.DataFrame:
    """
    Simulate GBM terminal prices with antithetic normal variates.
    """
    local_rng = np.random.default_rng(seed)

    z_half = local_rng.standard_normal(n_pairs)
    z = np.concatenate([z_half, -z_half])

    log_return = (
        (config.risk_free_rate - config.dividend_yield - 0.5 * config.volatility ** 2) * config.maturity
        + config.volatility * sqrt(config.maturity) * z
    )

    terminal_price = config.s0 * np.exp(log_return)

    return pd.DataFrame({
        "terminal_price": terminal_price,
        "z": z
    })


antithetic_terminal = simulate_gbm_terminal_antithetic(
    config=config,
    n_pairs=100_000,
    seed=42
)

antithetic_call = monte_carlo_price_from_terminal(
    terminal_price=antithetic_terminal["terminal_price"].to_numpy(),
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    option_type="call"
)

plain_terminal_same_n = simulate_gbm_terminal_prices(
    config=config,
    n_paths=len(antithetic_terminal),
    seed=42
)

plain_call_same_n = monte_carlo_price_from_terminal(
    terminal_price=plain_terminal_same_n["terminal_price"].to_numpy(),
    strike=config.strike,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    option_type="call"
)

pd.DataFrame([
    {"method": "plain", **plain_call_same_n, "error_vs_bsm": plain_call_same_n["price"] - bsm_call_reference},
    {"method": "antithetic", **antithetic_call, "error_vs_bsm": antithetic_call["price"] - bsm_call_reference},
])

## 12. Control variate using the stock price

For a call payoff, a natural control variate is the terminal stock price $S_T$, whose expectation is known:

$$
\mathbb E^{\mathbb Q}[S_T] = S_0e^{(r-q)T}
$$
Let:

$$
X = e^{-rT}g(S_T)
$$
and:

$$
Y = e^{-rT}S_T
$$
The expectation of $Y$ is known:

$$
\mathbb E[Y] = S_0e^{-qT}
$$
A control variate estimator is:

$$
X^{CV} = X - b(Y-\mathbb E[Y])
$$
where the optimal coefficient is approximately:

$$
b^\star = \frac{\operatorname{Cov}(X,Y)}{\operatorname{Var}(Y)}
$$

In [ ]:
def monte_carlo_call_with_stock_control_variate(
    terminal_price: np.ndarray,
    config: MonteCarloConfig
) -> dict:
    """
    Price a call using terminal stock as a control variate.
    """
    payoff = np.maximum(terminal_price - config.strike, 0.0)
    discounted_payoff = np.exp(-config.risk_free_rate * config.maturity) * payoff
    discounted_stock = np.exp(-config.risk_free_rate * config.maturity) * terminal_price

    expected_discounted_stock = config.s0 * np.exp(-config.dividend_yield * config.maturity)

    covariance = np.cov(discounted_payoff, discounted_stock, ddof=1)[0, 1]
    variance_control = np.var(discounted_stock, ddof=1)

    b_star = covariance / variance_control

    adjusted_payoff = discounted_payoff - b_star * (discounted_stock - expected_discounted_stock)

    price = float(np.mean(adjusted_payoff))
    standard_error = float(np.std(adjusted_payoff, ddof=1) / np.sqrt(len(adjusted_payoff)))

    plain_price = float(np.mean(discounted_payoff))
    plain_se = float(np.std(discounted_payoff, ddof=1) / np.sqrt(len(discounted_payoff)))

    return {
        "plain_price": plain_price,
        "plain_standard_error": plain_se,
        "control_variate_price": price,
        "control_variate_standard_error": standard_error,
        "b_star": float(b_star),
        "variance_reduction_ratio": float((plain_se ** 2) / (standard_error ** 2))
    }


control_variate_result = monte_carlo_call_with_stock_control_variate(
    terminal_price=gbm_terminal["terminal_price"].to_numpy(),
    config=config
)

pd.Series(control_variate_result)

## 13. Fat-tailed model A: finite normal mixture shocks

A simple fat-tailed model is a mixture of normal shocks.

Let:

$$
X =
\begin{cases}
\sigma_{\text{low}}\sqrt{T}Z, & \text{with probability } 1-p \\
\sigma_{\text{high}}\sqrt{T}Z, & \text{with probability } p
\end{cases}
$$
where $Z\sim\mathcal N(0,1)$.

This creates occasional high-volatility states.

The exponential moment is finite:

$$
\begin{aligned}
\mathbb E[e^X] &= (1-p)e^{\frac{1}{2}\sigma_{\text{low}}^2T} \\
&\quad + p e^{\frac{1}{2}\sigma_{\text{high}}^2T}
\end{aligned}
$$
So we can define a martingale-corrected terminal stock price:

$$
\begin{aligned}
S_T &= S_0 \exp \left[ (r-q)T + X - \log\mathbb E(e^X) \right]
\end{aligned}
$$

In [ ]:
@dataclass(frozen=True)
class NormalMixtureConfig:
    """
    Finite normal mixture shock model.
    """
    low_vol: float = 0.16
    high_vol: float = 0.55
    high_vol_probability: float = 0.08


mixture_config = NormalMixtureConfig()
mixture_config

In [ ]:
def simulate_normal_mixture_terminal_prices(
    base_config: MonteCarloConfig,
    mixture_config: NormalMixtureConfig,
    n_paths: int,
    seed: int = 42
) -> pd.DataFrame:
    """
    Simulate martingale-corrected terminal prices with finite normal mixture shocks.
    """
    local_rng = np.random.default_rng(seed)

    high_state = local_rng.uniform(size=n_paths) < mixture_config.high_vol_probability
    z = local_rng.standard_normal(n_paths)

    shock_vol = np.where(high_state, mixture_config.high_vol, mixture_config.low_vol)
    shock = shock_vol * sqrt(base_config.maturity) * z

    exponential_moment = (
        (1 - mixture_config.high_vol_probability)
        * exp(0.5 * mixture_config.low_vol ** 2 * base_config.maturity)
        + mixture_config.high_vol_probability
        * exp(0.5 * mixture_config.high_vol ** 2 * base_config.maturity)
    )

    martingale_correction = log(exponential_moment)

    log_return = (
        (base_config.risk_free_rate - base_config.dividend_yield) * base_config.maturity
        + shock
        - martingale_correction
    )

    terminal_price = base_config.s0 * np.exp(log_return)

    return pd.DataFrame({
        "terminal_price": terminal_price,
        "log_return": log_return,
        "shock": shock,
        "high_vol_state": high_state,
        "martingale_correction": martingale_correction
    })


mixture_terminal = simulate_normal_mixture_terminal_prices(
    base_config=config,
    mixture_config=mixture_config,
    n_paths=config.n_paths,
    seed=43
)

martingale_check_mixture = martingale_check_terminal(
    terminal_price=mixture_terminal["terminal_price"].to_numpy(),
    spot=config.s0,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield
)

martingale_check_mixture

## 14. Fat-tailed model B: truncated Student-t stress shocks

A raw Student-t log-return shock has very heavy tails and does not have a finite exponential moment.

That is a problem for risk-neutral stock-price modelling.

For stress testing, we can use a **truncated** Student-t shock:

1. sample Student-t innovations;
2. standardise them to unit variance;
3. clip extreme values;
4. scale by volatility;
5. apply an empirical martingale correction:

$$
m_X \approx \log \left(\frac{1}{M}\sum_{i=1}^{M}e^{X_i}\right)
$$
This is not a canonical pricing model. It is a controlled stress-simulation device.

In [ ]:
@dataclass(frozen=True)
class TruncatedStudentTConfig:
    """
    Truncated Student-t shock model.
    """
    df: float = 5.0
    annual_vol: float = 0.22
    clip_z: float = 6.0


student_config = TruncatedStudentTConfig()
student_config

In [ ]:
def simulate_truncated_student_terminal_prices(
    base_config: MonteCarloConfig,
    student_config: TruncatedStudentTConfig,
    n_paths: int,
    seed: int = 44
) -> pd.DataFrame:
    """
    Simulate terminal prices using truncated Student-t shocks with empirical martingale correction.
    """
    local_rng = np.random.default_rng(seed)

    raw = local_rng.standard_t(df=student_config.df, size=n_paths)

    # Standard Student-t variance is df/(df-2) for df > 2.
    standardised = raw / sqrt(student_config.df / (student_config.df - 2.0))
    clipped = np.clip(standardised, -student_config.clip_z, student_config.clip_z)

    shock = student_config.annual_vol * sqrt(base_config.maturity) * clipped

    # Empirical correction on the simulated sample.
    martingale_correction = float(np.log(np.mean(np.exp(shock))))

    log_return = (
        (base_config.risk_free_rate - base_config.dividend_yield) * base_config.maturity
        + shock
        - martingale_correction
    )

    terminal_price = base_config.s0 * np.exp(log_return)

    return pd.DataFrame({
        "terminal_price": terminal_price,
        "log_return": log_return,
        "raw_t": raw,
        "standardised_t": standardised,
        "clipped_t": clipped,
        "shock": shock,
        "martingale_correction": martingale_correction
    })


student_terminal = simulate_truncated_student_terminal_prices(
    base_config=config,
    student_config=student_config,
    n_paths=config.n_paths,
    seed=44
)

martingale_check_student = martingale_check_terminal(
    terminal_price=student_terminal["terminal_price"].to_numpy(),
    spot=config.s0,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    dividend_yield=config.dividend_yield
)

martingale_check_student

## 15. Return distribution comparison

We compare terminal log-return distributions under:

1. GBM Gaussian shocks;
2. finite normal mixture shocks;
3. truncated Student-t shocks.

The fat-tailed models should show larger tail quantiles and higher kurtosis.

In [ ]:
def distribution_diagnostics(series: pd.Series) -> pd.Series:
    """
    Basic return distribution diagnostics.
    """
    x = series.dropna().to_numpy()
    mean = np.mean(x)
    std = np.std(x, ddof=1)
    centered = x - mean

    skew = np.mean(centered ** 3) / (std ** 3)
    excess_kurtosis = np.mean(centered ** 4) / (std ** 4) - 3

    return pd.Series({
        "mean": mean,
        "std": std,
        "skew": skew,
        "excess_kurtosis": excess_kurtosis,
        "p001": np.quantile(x, 0.001),
        "p01": np.quantile(x, 0.01),
        "p05": np.quantile(x, 0.05),
        "p50": np.quantile(x, 0.50),
        "p95": np.quantile(x, 0.95),
        "p99": np.quantile(x, 0.99),
        "p999": np.quantile(x, 0.999),
    })


distribution_table = pd.DataFrame({
    "gbm_gaussian": distribution_diagnostics(gbm_terminal["log_return"]),
    "normal_mixture": distribution_diagnostics(mixture_terminal["log_return"]),
    "truncated_student_t": distribution_diagnostics(student_terminal["log_return"])
})

distribution_table

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(gbm_terminal["log_return"].clip(-1, 1), bins=120, density=True, alpha=0.5, label="GBM Gaussian")
plt.hist(mixture_terminal["log_return"].clip(-1, 1), bins=120, density=True, alpha=0.5, label="Normal mixture")
plt.hist(student_terminal["log_return"].clip(-1, 1), bins=120, density=True, alpha=0.5, label="Truncated Student-t")
plt.title("Terminal Log-Return Distributions")
plt.xlabel("Log return, clipped for display")
plt.ylabel("Density")
plt.legend()
plt.show()

## 16. Tail risk: VaR and Expected Shortfall

For a loss variable $L$, Value-at-Risk at confidence level $\alpha$ is:

$$
VaR_\alpha = \inf \{x : \mathbb P(L \leq x) \geq \alpha\}
$$
Expected Shortfall is:

$$
ES_\alpha = \mathbb E[L \mid L \geq VaR_\alpha]
$$
For terminal log returns, define loss as:

$$
L = -\log(S_T/S_0)
$$
Fat-tailed models should produce larger VaR and ES.

In [ ]:
def var_expected_shortfall_from_log_returns(
    log_returns: pd.Series,
    alpha: float
) -> pd.Series:
    """
    Compute VaR and expected shortfall from log returns using loss = -return.
    """
    losses = -log_returns.dropna().to_numpy()
    var = np.quantile(losses, alpha)
    es = losses[losses >= var].mean()

    return pd.Series({
        "alpha": alpha,
        "VaR": var,
        "ExpectedShortfall": es
    })


tail_rows = []

for name, series in [
    ("gbm_gaussian", gbm_terminal["log_return"]),
    ("normal_mixture", mixture_terminal["log_return"]),
    ("truncated_student_t", student_terminal["log_return"]),
]:
    for alpha in [0.95, 0.99, 0.995]:
        row = var_expected_shortfall_from_log_returns(series, alpha)
        tail_rows.append({
            "model": name,
            **row.to_dict()
        })

tail_risk_table = pd.DataFrame(tail_rows)

tail_risk_table

In [ ]:
plot_tail = tail_risk_table[tail_risk_table["alpha"] == 0.99].copy()

plt.figure(figsize=(10, 6))
plt.bar(plot_tail["model"], plot_tail["ExpectedShortfall"])
plt.title("99% Expected Shortfall by Terminal Return Model")
plt.xlabel("Model")
plt.ylabel("Expected shortfall of log-return loss")
plt.xticks(rotation=20, ha="right")
plt.show()

## 17. Option prices under fat-tailed terminal models

For each terminal model, we price vanilla calls and puts.

Fat-tailed downside models generally increase out-of-the-money put prices and can also affect calls through the martingale correction and tail shape.

Important: these are model prices under the simulated terminal distribution, not Black-Scholes prices.

In [ ]:
def price_option_under_models(
    terminal_data: dict[str, pd.DataFrame],
    strikes: np.ndarray,
    maturity: float,
    risk_free_rate: float,
    option_types: list[str]
) -> pd.DataFrame:
    """
    Price vanilla options using terminal distributions from multiple models.
    """
    rows = []

    for model_name, df in terminal_data.items():
        ST = df["terminal_price"].to_numpy()

        for strike in strikes:
            for option_type in option_types:
                result = monte_carlo_price_from_terminal(
                    terminal_price=ST,
                    strike=float(strike),
                    maturity=maturity,
                    risk_free_rate=risk_free_rate,
                    option_type=option_type
                )

                rows.append({
                    "model": model_name,
                    "strike": float(strike),
                    "option_type": option_type,
                    "price": result["price"],
                    "standard_error": result["standard_error"],
                    "lower_95": result["lower_95"],
                    "upper_95": result["upper_95"]
                })

    return pd.DataFrame(rows)


terminal_models = {
    "gbm_gaussian": gbm_terminal,
    "normal_mixture": mixture_terminal,
    "truncated_student_t": student_terminal
}

strike_grid = np.linspace(60, 140, 17)

option_price_table = price_option_under_models(
    terminal_data=terminal_models,
    strikes=strike_grid,
    maturity=config.maturity,
    risk_free_rate=config.risk_free_rate,
    option_types=["call", "put"]
)

option_price_table.head()

In [ ]:
plt.figure(figsize=(10, 6))

for model_name, group in option_price_table[option_price_table["option_type"] == "put"].groupby("model"):
    plt.plot(group["strike"], group["price"], marker="o", label=model_name)

plt.title("Put Prices under GBM and Fat-Tailed Terminal Models")
plt.xlabel("Strike")
plt.ylabel("Put price")
plt.legend()
plt.show()

## 18. BSM implied vols from fat-tailed model prices

A fat-tailed terminal model can be translated into a BSM implied-volatility smile.

For each strike:

1. compute the model option price by Monte Carlo;
2. solve for the BSM volatility that reproduces that price.

This shows how non-Gaussian risk appears when viewed through the Black-Scholes coordinate system.

In [ ]:
def implied_vol_bisection(
    target_price: float,
    spot: float,
    strike: float,
    maturity: float,
    risk_free_rate: float,
    dividend_yield: float,
    option_type: str,
    low: float = 1e-6,
    high: float = 5.0,
    tolerance: float = 1e-8,
    max_iter: int = 200
) -> float | None:
    """
    Simple BSM implied volatility solver using bisection.
    """
    discounted_spot = spot * exp(-dividend_yield * maturity)
    discounted_strike = strike * exp(-risk_free_rate * maturity)

    if option_type == "call":
        lower = max(discounted_spot - discounted_strike, 0.0)
        upper = discounted_spot
    elif option_type == "put":
        lower = max(discounted_strike - discounted_spot, 0.0)
        upper = discounted_strike
    else:
        raise ValueError("option_type must be 'call' or 'put'.")

    if target_price < lower - 1e-8 or target_price > upper + 1e-8:
        return None

    def objective(vol):
        return (
            bsm_price(
                spot=spot,
                strike=strike,
                maturity=maturity,
                risk_free_rate=risk_free_rate,
                dividend_yield=dividend_yield,
                volatility=vol,
                option_type=option_type
            )
            - target_price
        )

    f_low = objective(low)
    f_high = objective(high)

    if f_low * f_high > 0:
        return None

    for _ in range(max_iter):
        mid = 0.5 * (low + high)
        f_mid = objective(mid)

        if abs(f_mid) < tolerance or (high - low) < tolerance:
            return float(mid)

        if f_low * f_mid <= 0:
            high = mid
            f_high = f_mid
        else:
            low = mid
            f_low = f_mid

    return float(0.5 * (low + high))

In [ ]:
iv_rows = []

for row in option_price_table.itertuples():
    iv = implied_vol_bisection(
        target_price=float(row.price),
        spot=config.s0,
        strike=float(row.strike),
        maturity=config.maturity,
        risk_free_rate=config.risk_free_rate,
        dividend_yield=config.dividend_yield,
        option_type=str(row.option_type)
    )

    iv_rows.append({
        "model": row.model,
        "strike": row.strike,
        "option_type": row.option_type,
        "price": row.price,
        "bsm_implied_vol": iv
    })

iv_smile_table = pd.DataFrame(iv_rows)

iv_smile_table.head()

In [ ]:
plt.figure(figsize=(10, 6))

for model_name, group in iv_smile_table[iv_smile_table["option_type"] == "put"].groupby("model"):
    plt.plot(group["strike"], group["bsm_implied_vol"], marker="o", label=model_name)

plt.axhline(config.volatility, linestyle="--", label="GBM input volatility")
plt.title("Put-Implied Volatility Smiles from Terminal Models")
plt.xlabel("Strike")
plt.ylabel("BSM implied volatility")
plt.legend()
plt.show()

## 19. Model comparison dashboard

We summarise key diagnostics:

1. martingale error;
2. excess kurtosis;
3. 99% expected shortfall;
4. at-the-money call price;
5. out-of-the-money put price.

In [ ]:
def model_summary_table(
    terminal_models: dict[str, pd.DataFrame],
    distribution_table: pd.DataFrame,
    tail_risk_table: pd.DataFrame,
    option_price_table: pd.DataFrame
) -> pd.DataFrame:
    """
    Build compact model-comparison table.
    """
    rows = []

    for model_name, df in terminal_models.items():
        martingale = martingale_check_terminal(
            terminal_price=df["terminal_price"].to_numpy(),
            spot=config.s0,
            maturity=config.maturity,
            risk_free_rate=config.risk_free_rate,
            dividend_yield=config.dividend_yield
        )

        es99 = tail_risk_table[
            (tail_risk_table["model"] == model_name)
            & (tail_risk_table["alpha"] == 0.99)
        ]["ExpectedShortfall"].iloc[0]

        atm_call = option_price_table[
            (option_price_table["model"] == model_name)
            & (option_price_table["option_type"] == "call")
            & (option_price_table["strike"] == 100.0)
        ]["price"].iloc[0]

        otm_put = option_price_table[
            (option_price_table["model"] == model_name)
            & (option_price_table["option_type"] == "put")
            & (option_price_table["strike"] == 80.0)
        ]["price"].iloc[0]

        rows.append({
            "model": model_name,
            "discounted_martingale_error": martingale["discounted_martingale_error"],
            "excess_kurtosis": distribution_table.loc["excess_kurtosis", model_name],
            "expected_shortfall_99": es99,
            "atm_call_price_K100": atm_call,
            "otm_put_price_K80": otm_put
        })

    return pd.DataFrame(rows)


summary_table = model_summary_table(
    terminal_models=terminal_models,
    distribution_table=distribution_table,
    tail_risk_table=tail_risk_table,
    option_price_table=option_price_table
)

summary_table

## 20. Saving Monte Carlo outputs

The notebook saves:

1. GBM convergence table;
2. distribution diagnostics;
3. tail-risk table;
4. option price comparison;
5. implied-vol smile table;
6. model summary;
7. manifest.

These outputs feed later notebooks on exotic options, VaR/CVaR, volatility surfaces, and backtesting.

In [ ]:
output_dir = Path("data/processed/monte_carlo_gbm_and_fat_tails_v1")
output_dir.mkdir(parents=True, exist_ok=True)

convergence_path = output_dir / "gbm_call_convergence.csv"
distribution_path = output_dir / "terminal_distribution_diagnostics.csv"
tail_risk_path = output_dir / "tail_risk_var_es.csv"
option_prices_path = output_dir / "option_prices_by_terminal_model.csv"
iv_smile_path = output_dir / "bsm_implied_vols_from_terminal_models.csv"
summary_path = output_dir / "model_comparison_summary.csv"
manifest_path = output_dir / "manifest.json"

convergence_call.to_csv(convergence_path, index=False)
distribution_table.to_csv(distribution_path)
tail_risk_table.to_csv(tail_risk_path, index=False)
option_price_table.to_csv(option_prices_path, index=False)
iv_smile_table.to_csv(iv_smile_path, index=False)
summary_table.to_csv(summary_path, index=False)

manifest = {
    "dataset_name": "monte_carlo_gbm_and_fat_tails",
    "schema_version": "monte_carlo_gbm_and_fat_tails_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "mixture_config": asdict(mixture_config),
    "student_config": asdict(student_config),
    "bsm_call_reference": float(bsm_call_reference),
    "bsm_put_reference": float(bsm_put_reference),
    "notes": [
        "GBM terminal prices are simulated exactly under the risk-neutral measure.",
        "Normal mixture shocks are martingale-corrected analytically.",
        "Truncated Student-t shocks are used as a stress model with empirical martingale correction.",
        "Raw unbounded Student-t log shocks are not used directly as risk-neutral stock-price shocks because the exponential moment is not finite.",
        "Monte Carlo confidence intervals reflect sampling error, not model error."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

convergence_path, distribution_path, tail_risk_path, option_prices_path, manifest_path

## 21. Practical checklist for Monte Carlo pricing

Before trusting a Monte Carlo price, check:

1. **Pricing measure**  
   Are paths simulated under $\mathbb Q$, not $\mathbb P$, if the task is pricing?

2. **Martingale condition**  
   Does the simulation satisfy:

$$
e^{-rT}\mathbb E[S_T] \approx S_0e^{-qT}
$$
3. **Payoff definition**  
   Is the payoff implemented exactly as specified?

4. **Discounting**  
   Is the payoff discounted using the correct curve or rate?

5. **Standard error**  
   Is the confidence interval reported?

6. **Convergence**  
   Does increasing paths reduce error at the expected rate?

7. **Variance reduction**  
   Can antithetic, control variate, stratified, or quasi-Monte Carlo methods help?

8. **Random seed**  
   Is the simulation reproducible?

9. **Path discretisation**  
   For path-dependent products, is the time grid fine enough?

10. **Model risk**  
   Does the model capture tails, jumps, volatility clustering, and smile behaviour?

11. **Numerical stability**  
   Are exponentials, clipping, and martingale corrections handled carefully?

12. **Audit trail**  
   Are parameters, outputs, and diagnostics saved?

## 22. Limitations

### 22.1 GBM is too clean

GBM assumes:

- constant volatility;
- continuous paths;
- lognormal terminal distribution;
- no jumps;
- no stochastic volatility;
- no fat tails beyond lognormality.

This is rarely enough for realistic risk modelling.

### 22.2 Monte Carlo has sampling error

Unlike closed-form formulas, Monte Carlo prices are random estimates.

Reporting a Monte Carlo price without a standard error is incomplete.

### 22.3 Fat-tail models need care

Heavy-tailed return models used for risk simulation are not automatically valid for risk-neutral pricing.

The martingale condition and finite exponential moments matter.

### 22.4 Truncated Student-t is a stress device

The truncated Student-t model in this notebook is not a canonical no-arbitrage model.

It is included to demonstrate fat-tail stress simulation and martingale correction.

### 22.5 Terminal models are not path models

A terminal distribution can price European payoffs, but path-dependent products require a full stochastic process.

Different processes can share the same terminal distribution but produce different barrier or Asian option prices.

### 22.6 Variance reduction is payoff-dependent

Antithetic variates and control variates help some payoffs more than others.

They should be benchmarked rather than assumed.

### 22.7 Model error is not Monte Carlo error

A tiny confidence interval only means the simulation has converged under the chosen model.

It does not mean the model is correct.

## 23. Research frontier and current directions

Monte Carlo remains central in computational finance, but modern methods focus on speed, dimensionality, and model realism.

### 23.1 Quasi-Monte Carlo

Quasi-Monte Carlo uses low-discrepancy sequences instead of pseudo-random numbers.

It can dramatically improve convergence for some smooth, moderate-dimensional pricing problems.

A future notebook could compare pseudo-random Monte Carlo with Sobol sequences for European and Asian options.

### 23.2 Multilevel Monte Carlo

Multilevel Monte Carlo reduces cost by combining many cheap coarse simulations with fewer expensive fine simulations.

It is especially useful for path-dependent derivatives and SDE discretisation problems.

A future notebook could price an Asian option with standard Monte Carlo and multilevel Monte Carlo.

### 23.3 Monte Carlo Greeks

Greeks can be estimated by:

- finite differences;
- pathwise derivatives;
- likelihood-ratio methods;
- automatic differentiation;
- adjoint methods.

A future notebook could compare Delta estimators and their variance.

### 23.4 GPU and JAX Monte Carlo

Monte Carlo is highly parallel.

Modern implementations often use:

- CuPy;
- JAX;
- PyTorch;
- CUDA;
- C++;
- vectorised NumPy.

A future notebook could benchmark a batched Monte Carlo engine across CPU and GPU backends.

### 23.5 Deep Monte Carlo and neural pricing

Neural networks can approximate pricing functions across many strikes, maturities, and model parameters.

A future notebook could train a neural surrogate model to approximate Monte Carlo option prices.

### 23.6 Scenario generation and market generators

Modern risk systems increasingly use learned or simulated scenario generators.

The challenge is to generate realistic fat tails, dependence, volatility clustering, and no-arbitrage constraints.

A future notebook could compare GBM, jump diffusion, GARCH, and neural scenario generators.

### 23.7 Robust and distributionally robust pricing

Instead of trusting one model, robust pricing asks how prices change across a set of plausible distributions.

A future notebook could compute option price bounds under moment constraints or alternative tail assumptions.

## 24. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_07_exotic_options_monte_carlo**  
   Pricing Asian, barrier, and lookback options using full simulated paths.

2. **02_08_dynamic_delta_hedging_simulation**  
   Simulating discrete hedging error under GBM and fat-tailed alternatives.

3. **02_13_multilevel_monte_carlo_pricing**  
   Reducing simulation cost for path-dependent derivatives.

4. **03_01_garch_volatility_modeling**  
   Comparing fat-tailed terminal models with volatility clustering.

5. **04_06_var_cvar_and_stress_testing**  
   Using simulated fat tails for risk management.

6. **05_01_vectorized_backtest_engine**  
   Using Monte Carlo scenarios to stress strategy returns.

7. **06_11_cpp_python_bindings_pybind11**  
   Moving simulation kernels into compiled code.

8. **07_02_volatility_surface_pricing_and_alpha**  
   Connecting fat-tailed pricing to implied volatility smile behaviour.

## 25. Summary

This notebook implemented Monte Carlo simulation under GBM and fat-tailed terminal models.

It showed how to:

1. simulate exact terminal GBM prices;
2. price European options by discounted expected payoff;
3. report confidence intervals and standard errors;
4. verify the risk-neutral martingale condition;
5. study Monte Carlo convergence;
6. simulate full GBM paths;
7. apply antithetic variates and control variates;
8. generate finite normal-mixture fat tails;
9. build a truncated Student-t stress model;
10. compare VaR and expected shortfall;
11. compare option prices across terminal distributions;
12. extract BSM implied volatility smiles from fat-tailed model prices.

The key computational takeaway is:

> Monte Carlo pricing must report uncertainty, diagnostics, and model assumptions. A simulated price without error bars and martingale checks is incomplete.

The key financial takeaway is:

> Fat tails change both risk estimates and option prices, but heavy-tailed risk simulations must be handled carefully before being interpreted as arbitrage-free pricing models.

## 26. Further reading

### Monte Carlo foundations

- Glasserman, P. *Monte Carlo Methods in Financial Engineering.*
- Jäckel, P. *Monte Carlo Methods in Finance.*
- Shreve, S. E. *Stochastic Calculus for Finance II.*
- Hull, J. C. *Options, Futures, and Other Derivatives.*

### Variance reduction

- Antithetic variates.
- Control variates.
- Stratified sampling.
- Importance sampling.
- Quasi-Monte Carlo and Sobol sequences.

### Advanced Monte Carlo

- Giles, M. B. "Multilevel Monte Carlo Methods."
- Pathwise and likelihood-ratio Greeks.
- Adjoint algorithmic differentiation.
- GPU Monte Carlo simulation.
- JAX-based differentiable Monte Carlo.

### Fat tails and model risk

- Student-t and mixture distributions.
- Jump-diffusion models.
- Variance Gamma and Lévy models.
- GARCH and stochastic volatility.
- Distributionally robust pricing.